In [4]:
import pandas as pd

In [5]:
air_data = pd.read_csv('AirQuality_Daily_StudentVersion.csv')
air_data = pd.DataFrame(air_data)

air_data


,date,monitor_index,humidity,pressure,temperature,voc,analog_input,pm2.5_alt,pm1.0_atm,pm2.5_atm,pm10.0_atm,sensor.latitude,sensor.longitude,sensor.altitude,sensor.name
0,2/23/2024,195089,14.377667,912.884333,62.266667,51.998667,0.051333,0.100000,0.000000,0.002500,0.039667,40.050922,-101.533570,3005,Swnphd-Benklemen
1,2/23/2024,195365,12.223600,926.403000,71.193400,64.920800,0.000000,0.180000,0.004800,0.020000,0.176000,40.200330,-100.639885,2576,Swnphd-mccook
2,2/23/2024,195541,20.095750,905.670750,61.008250,68.307000,0.020000,0.162500,0.004125,0.014812,0.063937,41.128284,-101.720220,3220,Swnphd-ogallala
3,2/24/2024,195089,25.368000,911.708833,51.462458,91.176750,0.052667,0.437500,0.099542,0.170667,0.355208,40.050922,-101.533570,3005,Swnphd-Benklemen
4,2/24/2024,195365,23.703083,925.282125,56.818208,107.863708,0.000000,0.475000,0.099208,0.231687,0.548583,40.200330,-100.639885,2576,Swnphd-mccook
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8296,3/24/2025,195373,23.636208,915.998292,61.489083,421.493750,0.020000,0.762500,0.486583,0.830021,1.416125,41.148170,-100.761800,2800,WCDHD City Building
8297,3/24/2025,195379,28.043545,954.294546,66.009182,301.074800,0.000000,0.800000,5.625545,5.873364,6.056773,40.877007,-97.579445,1639,FCHD-YPS
8298,3/24/2025,195383,31.451375,936.556583,55.752792,353.478000,0.050000,0.575000,0.267500,0.478375,0.818146,41.779380,-99.138040,2174,Loup Basin Public Health Department
8299,3/24/2025,195385,NaN,NaN,NaN,NaN,0.010000,0.970833,0.739500,1.088604,1.340563,40.019714,-98.071840,1597,South Heartland District Health Dept. Superior...


In [85]:
voc = pd.pivot_table(
    air_data,
    index='sensor.name',
    values='voc',
    aggfunc=['median', 'mean']
).round(2)

top_5_voc_df = (
    voc
    .sort_values(('mean', 'voc'), ascending=False)
    .head(5)
)

top_5_voc_df






,median,mean
,voc,voc
sensor.name,,
Swnphd-ogallala,423.08,399.43
FCHD-YPS,375.38,372.46
Three Rivers Public Health Department,376.81,370.22
ELVPHD Norfolk HD 4,368.58,360.83
Swnphd-mccook,381.47,353.94


In [7]:

max_pm25_value = air_data['pm2.5_atm'].max()


max_pm25_events = air_data[
    air_data['pm2.5_atm'] == max_pm25_value
][['date', 'sensor.name', 'pm2.5_atm']]

max_pm25_events



,date,sensor.name,pm2.5_atm
7561,2/18/2025,#16 - Richardson County Courthouse,3782.823313


In [90]:
max_pm10_value = air_data['pm10.0_atm'].max()

max_pm10_events = air_data[
    air_data['pm10.0_atm'] == max_pm10_value
][['date', 'sensor.name', 'pm10.0_atm']]

max_pm10_events




,date,sensor.name,pm10.0_atm
7561,2/18/2025,#16 - Richardson County Courthouse,3784.682542


In [91]:
max_voc_value = air_data['voc'].max()

max_voc_events = air_data[
    air_data['voc'] == max_voc_value
][['date', 'sensor.name', 'voc']]

max_voc_events




,date,sensor.name,voc
2391,6/24/2024,Swnphd-ogallala,1209.931571


In [ ]:
unhealthy_voc = air_data[air_data['voc'] >= 25]



unhealthy_events = unhealthy_voc[
    ['date', 'sensor.name', 'voc']].sort_values('date')

unhealthy_events.head(10)

In [15]:
def assign_temp(temp):
    if temp < 32:
        return 'Below freezing'
    elif 32 <= temp <= 50:
        return 'Cool'
    elif 51 <= temp <= 70:
        return 'Warm'
    else:
        return 'Hot'

air_data['temp_level'] = air_data['temperature'].apply(assign_temp)


In [8]:
def assign_altitude(alt):
    if alt < 1000:
        return 'Low'
    elif 1000 <= alt <= 2000:
        return 'Medium'
    else:
        return 'High'

air_data['alt_level'] = air_data['sensor.altitude'].apply(assign_altitude)


In [9]:
air_data_grouped = air_data.groupby('sensor.name')

air_data_aggregated_pm2_5 = air_data_grouped.agg(
    max_pm2_5=('pm2.5_atm', 'max'),
    min_pm2_5=('pm2.5_atm', 'min'),
    mean_pm2_5=('pm2.5_atm', 'mean'),
    median_pm2_5=('pm2.5_atm', 'median')
)

top5_mean_pm2_5 = (
    air_data_aggregated_pm2_5
    .sort_values(by='mean_pm2_5', ascending=False)
    .head(5)
)

top5_mean_pm2_5


,max_pm2_5,min_pm2_5,mean_pm2_5,median_pm2_5
sensor.name,,,,
Broken Bow,3205.868854,0.019187,928.710593,36.050240
#16 - Richardson County Courthouse,3782.823313,0.243729,700.127342,11.977344
#18 - Southeast District Health Department- Tecumseh,2987.467563,0.140708,613.175352,10.322875
NCDHD O'Neill #11,2078.873729,0.136958,164.495078,7.251208
Swnphd-mccook,3053.114292,0.020000,123.011622,4.582281


In [30]:
air_data_grouped_monitor = air_data.groupby('monitor_index')

air_data_aggregated_voc = air_data_grouped_monitor.agg(
    mean_voc=('voc', 'mean'),
    median_voc=('voc', 'median')
)

top10_max_voc = (
    air_data_aggregated_voc
    .sort_values(by='mean_voc', ascending=False)
    .head(10)
)

top10_max_voc


,mean_voc,median_voc
monitor_index,,
195541,399.434240,423.082292
195379,372.462720,375.383500
195367,370.216208,376.810167
195343,360.833744,368.580500
195365,353.941581,381.468479
195383,352.833902,343.790125
195089,336.400969,355.171583
195345,328.835391,343.764625
195333,311.912841,321.994833


In [19]:
unhealthy_voc = air_data[air_data['voc'] >= 25]

unhealthy_events = unhealthy_voc[['date', 'sensor.name', 'voc']].sort_values('date')
unhealthy_events.head(10)


,date,sensor.name,voc
6567,1/1/2025,Three Rivers Public Health Department,268.037042
6571,1/1/2025,Swnphd-ogallala,522.399333
6570,1/1/2025,South Heartland District Health Dept. Superior...,199.845625
6569,1/1/2025,Loup Basin Public Health Department,193.350875
6568,1/1/2025,FCHD-YPS,232.136042
6566,1/1/2025,Swnphd-mccook,440.842000
6565,1/1/2025,Broken Bow,236.872083
6552,1/1/2025,SWNPHD-Imerial,377.393125
6563,1/1/2025,Ainsworth Public School #9,310.367708
6564,1/1/2025,WCDHD Arthur High School 28,380.885625


In [20]:
warm_temp = air_data[air_data['temp_level'] == 'Warm']

warm_temp_altitude = warm_temp.groupby('sensor.altitude')

warm_temp_median = warm_temp_altitude.agg(
    median_voc=('voc', 'median')
)

top3_voc_warm_altitude = (
    warm_temp_median
    .sort_values(by='median_voc', ascending=False)
    .head(3)
)

top3_voc_warm_altitude


,median_voc
sensor.altitude,
3661,360.482708
3220,349.289333
1055,322.308333


In [31]:
correct_air_groups = air_data[air_data['temperature'] >= 50].groupby('sensor.name')

correct_air_summary_voc = correct_air_groups.agg(
    mean=('voc', 'mean'),
    median=('voc', 'median')
)

correct_air_summary_pm25 = correct_air_groups.agg(
    mean=('pm2.5_atm', 'mean'),
    median=('pm2.5_atm', 'median')
)

correct_air_summary_voc.head()


,mean,median
sensor.name,,
#16 - Richardson County Courthouse,86.758650,84.470500
#17 - Otoe County,224.985400,180.043187
#18 - Southeast District Health Department- Tecumseh,176.480209,119.176812
Ainsworth Public School #9,216.051783,206.472917
Broken Bow,137.378782,125.500583


In [32]:
correct_air_groups2 = air_data[
    (air_data['humidity'] >= 20) & (air_data['sensor.altitude'] >= 3000)
].groupby('monitor_index')

correct_air_summary_pm10 = correct_air_groups2.agg(
    mean=('pm10.0_atm', 'mean'),
    median=('pm10.0_atm', 'median')
)

correct_air_summary_pm10.head()


,mean,median
monitor_index,,
194969,7.347933,3.935271
195089,9.194444,5.659677
195355,7.687824,4.660073
195541,11.765493,4.388208


In [23]:
pivot_pm25 = pd.pivot_table(
    air_data,
    index='sensor.name',
    values='pm2.5_atm',
    aggfunc=['count', 'mean', 'max']
).round(2)

pivot_pm25


,count,mean,max
,pm2.5_atm,pm2.5_atm,pm2.5_atm
sensor.name,,,
#16 - Richardson County Courthouse,318,700.13,3782.82
#17 - Otoe County,302,9.76,37.25
#18 - Southeast District Health Department- Tecumseh,183,613.18,2987.47
Ainsworth Public School #9,368,10.70,43.32
Broken Bow,368,928.71,3205.87
Buffalo County TRPHD #26,359,8.48,42.23
ELVPHD Norfolk HD 4,171,13.37,79.97
ELVPHD Tekamah HD 3,202,9.48,45.88


In [49]:
pivot_pm10 = pd.pivot_table(
    air_data,
    index='sensor.name',
    values='pm10.0_atm',
    aggfunc=['median', 'mean']
).round(2)

top_5_pm10_df = (
    pivot_pm10
    .sort_values(('mean', 'pm10.0_atm'), ascending=False)
    .head(5)
)
top_5_pm10_df



#pivot_pm10


,median,mean
,pm10.0_atm,pm10.0_atm
sensor.name,,
Broken Bow,43.18,929.68
#16 - Richardson County Courthouse,13.31,701.63
#18 - Southeast District Health Department- Tecumseh,11.43,614.23
NCDHD O'Neill #11,8.43,166.13
Swnphd-mccook,5.37,124.23
